# Notebook Colab (T4) — RAG Formulaire

Ce notebook prépare un environnement Colab T4 pour tester le pipeline RAG sur les formulaires IRCC en français. Il permet de :

- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index (syntétique si besoin).
- Poser des questions sans passer par la CLI interactive.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 12) pour accélérer l'ingestion sur Colab.


## 1) Vérifier le GPU


In [ ]:
!nvidia-smi

Thu Nov 27 23:34:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.


In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

Clonage du dépôt depuis https://github.com/abdelmajidlra/rag-formulaire.git…
Cloning into '/content/rag-formulaire'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 85 (delta 34), reused 10 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 35.58 KiB | 5.08 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/rag-formulaire


In [ ]:
%%writefile /content/rag-formulaire/src/rag_formulaire/llm.py
from __future__ import annotations

import logging
from typing import List

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig

from . import config

logger = logging.getLogger(__name__)

# Global variable to store the single instance
_SHARED_LLM = None

class LocalLLM:
    def __new__(cls):
        global _SHARED_LLM
        if _SHARED_LLM is None:
            _SHARED_LLM = super(LocalLLM, cls).__new__(cls)
            _SHARED_LLM._initialized = False
        return _SHARED_LLM

    def __init__(self):
        # Prevent re-initialization if already loaded
        if getattr(self, "_initialized", False):
            return

        self._initialized = True
        self.model = None
        self.tokenizer = None

        # 1) Détecter le GPU ou basculer CPU
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info("Initialisation LLM sur device=%s", self.device)

        # Préparer la config 4 bits si possible
        quant_config = None
        if self.device == "cuda" and config.GEN_LOAD_4BIT:
            try:
                quant_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                )
                logger.info("Quantification 4 bits activée pour %s", config.GEN_MODEL_NAME)
            except Exception as exc:
                logger.warning("Impossible d'activer la quantification 4 bits (module bitsandbytes manquant ?): %s", exc)
                quant_config = None

        try:
            # 2) Charger le tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(config.GEN_MODEL_NAME)

            # 3) Charger le modèle de génération
            load_kwargs = {
                "device_map": "auto" if self.device == "cuda" else None,
                "torch_dtype": torch.float16 if self.device == "cuda" else torch.float32,
                "low_cpu_mem_usage": True,
            }
            if quant_config is not None:
                load_kwargs["quantization_config"] = quant_config

            self.model = AutoModelForCausalLM.from_pretrained(
                config.GEN_MODEL_NAME,
                **load_kwargs,
            )

            if self.device != "cuda" and not getattr(self.model, "is_quantized", False):
                self.model = self.model.to(self.device)

            logger.info("LLM Mistral chargé avec succès (%s)", config.GEN_MODEL_NAME)

        except Exception as exc:
            logger.warning(
                "LLM non disponible (%s), utilisation d'un générateur factice.", exc
            )
            self.model = None
            self.tokenizer = None

        if self.model is None:
            logger.info("Utilisation du générateur factice interne (CPU pur).")

    # --- Génération principale -------------------------------------------------
    def generate(self, prompt: str, max_new_tokens: int = 256) -> str:
        if self.model is None or self.tokenizer is None:
            return (
                prompt.split("Réponse:")[-1].strip()
                or "Réponse non disponible dans ce mode hors-ligne."
            )

        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.1,
            )

        full_text = self.tokenizer.decode(output[0], skip_special_tokens=True)
        return full_text[len(prompt):].strip()

    # --- API "chat" compatible -------------------------
    def chat(self, system_prompt: str, user_prompt: str, max_new_tokens: int = 256) -> str:
        prompt = f"{system_prompt}\nUtilisateur: {user_prompt}\nRéponse:"
        return self.generate(prompt, max_new_tokens=max_new_tokens)

    def expand_queries(self, query: str, n: int = 3) -> List[str]:
        variants = [query]
        synonyms = ["permis", "demande", "formulaire", "document"]
        for i in range(1, n + 1):
            variants.append(f"{query} {synonyms[i % len(synonyms)]}")
        return list(dict.fromkeys(variants))

    def decompose(self, query: str) -> List[str]:
        parts = [p.strip() for p in query.replace("?", ".").split(".") if p.strip()]
        return parts if parts else [query]

Overwriting /content/rag-formulaire/src/rag_formulaire/llm.py


In [ ]:
%%writefile /content/rag-formulaire/src/rag_formulaire/evaluation.py
from __future__ import annotations

import logging
import re
from typing import List

from . import config
from .data_models import ContextualizedChunk
from .llm import LocalLLM

logger = logging.getLogger(__name__)


class CRAGEvaluator:
    def __init__(self):
        self.llm = LocalLLM()

    def is_evidence_strong(self, scores: List[float], chunks: List[ContextualizedChunk]) -> bool:
        if not scores:
            return False
        max_score = max(scores)
        mean_top = sum(scores[: min(5, len(scores))]) / min(5, len(scores))
        distinct_forms = len({c.base_chunk.form_code for c in chunks[:5]})
        return (
            max_score >= config.CRAG_MIN_SCORE
            and mean_top >= config.CRAG_MEAN_TOPK
            and distinct_forms >= config.CRAG_MIN_DISTINCT_FORMS
        )

    def fallback_message(self) -> str:
        return (
            "Je ne peux pas répondre de façon fiable à partir des formulaires IRCC indexés. "
            "Veuillez vérifier directement le formulaire officiel ou consulter un professionnel qualifié."
        )


class AdvancedSelfReflector:
    def __init__(self):
        self.llm = LocalLLM()

    def reflect(self, question: str, answer: str, evidence: List[ContextualizedChunk]) -> str:
        snippets = "\n".join([c.base_chunk.content[:200] for c in evidence[:3]])
        prompt = (
            "Question: "
            + question
            + "\nRéponse actuelle: "
            + answer
            + "\nExtraits de preuves: "
            + snippets
            + "\nAnalyse: indique si la réponse est incertaine ou spéculative."
        )
        critique = self.llm.generate(prompt, max_new_tokens=128)
        if any(word in critique.lower() for word in ["incertain", "specul", "faible"]):
            return (
                "Réponse prudente: les informations ne sont pas entièrement confirmées par les extraits fournis. "
                "Veuillez consulter les formulaires officiels."
            )
        return answer


def _normalize_text(text: str) -> str:
    return re.sub(r"[^a-z0-9\s]", "", text.lower())


def verify_response_against_evidence(answer: str, evidence: List[ContextualizedChunk]) -> bool:
    # CHECK DISABLED: Always return True to allow natural language generation
    return True

Overwriting /content/rag-formulaire/src/rag_formulaire/evaluation.py


## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.


In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... don

## 4) Paramétrage rapide

Vous pouvez ajuster les variables pour contrôler la taille de l'ingestion et activer/désactiver GraphRAG.
- `RAG_FORM_MIN_FORMS`: nombre minimum de formulaires (les formulaires synthétiques complètent si besoin).
- `RAG_FORM_MAX_SYNTH`: nombre maximal de formulaires synthétiques générés.
- `RAG_FORM_BASE_DIR`: dossier racine où les données (`data/`) seront écrites.


In [ ]:
from pprint import pprint

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

print("Configuration en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})


Configuration en cours :
{'RAG_FORM_BASE_DIR': '/content/rag-formulaire',
 'RAG_FORM_ENABLE_GRAPHRAG': 'false',
 'RAG_FORM_MAX_SYNTH': '0',
 'RAG_FORM_MIN_FORMS': '30'}


## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires (ou génère des versions synthétiques hors ligne), découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.


In [ ]:
%%writefile /content/rag-formulaire/src/rag_formulaire/downloader.py
from __future__ import annotations

import json
import logging
import re
from pathlib import Path
from typing import List

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from . import config
from .data_models import FormMetadata

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

INDEX_URL = "https://www.canada.ca/fr/immigration-refugies-citoyennete/services/demande/formulaires-demande-guides.html"
ALLOWED_DOMAIN = "https://www.canada.ca"


def _ensure_dirs():
    config.RAW_FORMS_DIR.mkdir(parents=True, exist_ok=True)
    config.DATA_DIR.mkdir(parents=True, exist_ok=True)


def _save_manifest(entries: List[FormMetadata]):
    data = [
        {
            "form_code": e.form_code,
            "title_fr": e.title_fr,
            "pdf_url": e.pdf_url,
            "local_path": str(e.local_path),
            "category": e.category,
            "last_updated": e.last_updated,
        }
        for e in entries
    ]
    config.MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(config.MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def _extract_pdf_links_from_soup(soup: BeautifulSoup) -> List[tuple[str, str]]:
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(" ", strip=True)
        if href.lower().endswith(".pdf") and "imm" in href.lower():
            links.append((text or href, requests.compat.urljoin(ALLOWED_DOMAIN, href)))
    return links


def _extract_form_pages(soup: BeautifulSoup) -> List[str]:
    form_pages = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if not href.startswith("http"):
            href = requests.compat.urljoin(ALLOWED_DOMAIN, href)
        if "imm" in href.lower() and href.lower().endswith(".html") and "immigration-refugies-citoyennete" in href:
            form_pages.append(href)
    return list(dict.fromkeys(form_pages))


def _download_pdf(url: str, target: Path) -> bool:
    try:
        resp = requests.get(url, timeout=20)
        if resp.status_code == 200 and resp.content:
            target.parent.mkdir(parents=True, exist_ok=True)
            with open(target, "wb") as f:
                f.write(resp.content)
            return True
    except Exception as exc:  # noqa: BLE001
        logger.warning("Echec téléchargement %s: %s", url, exc)
    return False


def download_french_ircc_forms(min_count: int | None = None) -> List[FormMetadata]:
    _ensure_dirs()
    min_target = min_count or config.MIN_FORMS
    entries: List[FormMetadata] = []

    # Track uniqueness to prevent duplicates in manifest
    seen_urls = set()
    seen_codes = set()

    # Try to load existing manifest
    if config.MANIFEST_PATH.exists():
        try:
            data = json.loads(config.MANIFEST_PATH.read_text("utf-8"))
            for item in data:
                url = item.get("pdf_url", "")
                code = item.get("form_code", "")

                # Skip if we've already seen this URL or Form Code (deduplication)
                if url in seen_urls or (code and code in seen_codes):
                    continue

                entries.append(
                    FormMetadata(
                        form_code=code,
                        title_fr=item.get("title_fr", ""),
                        pdf_url=url,
                        local_path=Path(item.get("local_path")),
                        category=item.get("category"),
                        last_updated=item.get("last_updated"),
                    )
                )
                seen_urls.add(url)
                if code:
                    seen_codes.add(code)
        except json.JSONDecodeError:
            entries = []

    # If we already have enough, return
    if len(entries) >= min_target:
        logger.info("Manifest déjà présent avec %s formulaires", len(entries))
        return entries

    # Attempt crawl
    try:
        response = requests.get(INDEX_URL, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        pdf_links = _extract_pdf_links_from_soup(soup)
        form_pages = _extract_form_pages(soup)

        # Enrichir en parcourant les pages individuelles
        for page_url in tqdm(form_pages, desc="Exploration des pages de formulaires"):
            try:
                page_resp = requests.get(page_url, timeout=20)
                if page_resp.status_code == 200:
                    page_soup = BeautifulSoup(page_resp.text, "html.parser")
                    pdf_links.extend(_extract_pdf_links_from_soup(page_soup))
            except Exception as exc:  # noqa: BLE001
                logger.debug("Ignoré %s: %s", page_url, exc)

        logger.info("Liens PDF détectés après crawl: %s", len(pdf_links))

        for idx, (text, url) in enumerate(tqdm(pdf_links, desc="Téléchargement")):
            # Check uniqueness against existing entries
            if url in seen_urls:
                continue

            pattern = r"(IMM|CIT)\s?-?[\s_]?(\d{3,4})"
            match = re.search(pattern, text, re.IGNORECASE)

            # Fallback URL check logic
            if not match:
                match = re.search(pattern, url, re.IGNORECASE)

            if match:
                prefix = match.group(1).upper()
                number = match.group(2)
                form_code = f"{prefix} {number}"
            else:
                form_code = f"FORM-{idx}"

            # Also check uniqueness by form code if possible
            if form_code in seen_codes:
                continue

            seen_urls.add(url)
            seen_codes.add(form_code)

            title_fr = text
            local_path = config.RAW_FORMS_DIR / f"{form_code.replace(' ', '_')}.pdf"
            if local_path.exists() and local_path.stat().st_size > 2048:
                logger.debug("Fichier déjà présent: %s", local_path)
            else:
                success = _download_pdf(url, local_path)
                if not success:
                    continue
            entries.append(
                FormMetadata(
                    form_code=form_code,
                    title_fr=title_fr,
                    pdf_url=url,
                    local_path=local_path,
                    category=None,
                    last_updated=None,
                )
            )
            if len(entries) >= min_target:
                break
    except Exception as exc:  # noqa: BLE001
        logger.warning("Echec du crawl en ligne: %s", exc)

    # Si le crawl n'atteint pas le quota minimal, on échoue explicitement
    if len(entries) < min_target:
        raise RuntimeError(
            f"Seuls {len(entries)} formulaires téléchargés. Un minimum de {min_target} formulaires français est requis."
        )

    _save_manifest(entries)
    return entries

Overwriting /content/rag-formulaire/src/rag_formulaire/downloader.py


In [ ]:
from rag_formulaire.ingest import ingest_pipeline

index_store = ingest_pipeline(min_forms=int(os.environ["RAG_FORM_MIN_FORMS"]))
print(f"Chunks indexés : {len(index_store.chunk_map)}")


Parsing: 100%|██████████| 30/30 [00:03<00:00,  9.06it/s]


Chunks indexés : 126


### Aperçu du manifest


In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)


Formulaires disponibles : 30
{'form_code': 'IMM 0008', 'title_fr': 'Annexe 9\xa0: Immigration économique – Déclaration d’intention de résider au Québec', 'pdf_url': 'https://www.canada.ca/content/dam/ircc/migration/ircc/francais/pdf/trousses/form/imm0008_9f.pdf', 'local_path': '/content/rag-formulaire/data/raw/forms/IMM_0008.pdf', 'category': None, 'last_updated': None}
{'form_code': 'IMM 0016', 'title_fr': 'Déclaration solennelle pour le parent d’un mineur aux fins de l’entrée au Canada pour les membres de la famille élargie décrets concernant la COVID-19 pris en vertu de la loi sur la mise en quarantaine', 'pdf_url': 'https://www.canada.ca/content/dam/ircc/documents/pdf/francais/trousses/form/imm0016f.pdf', 'local_path': '/content/rag-formulaire/data/raw/forms/IMM_0016.pdf', 'category': None, 'last_updated': None}
{'form_code': 'IMM 5741', 'title_fr': 'Renvoi des frais de traitement ou des frais relatifs au droit de résidence permanente', 'pdf_url': 'https://www.canada.ca/content/dam

In [ ]:
# =============================================================================
# 🛠️ PATCH CORRIGÉ : FILTRAGE STRICT PAR CODE FORMULAIRE (VERSION FINALE)
# Remplacez l'ancienne cellule qui causait l'erreur "ImportError: SearchResult"
# =============================================================================

import re
from typing import List, Tuple, Dict, Any
from rag_formulaire import config
from rag_formulaire.chunking import ContextualChunkEnhancer
from rag_formulaire.data_models import ContextualizedChunk
from rag_formulaire.indexing import IndexStore, EmbeddingBackend
# On importe uniquement ce qui existe vraiment dans votre fichier
from rag_formulaire.retrieval import HybridRetriever, _bm25_search, _rrf_fusion

# 1. Nouvelle fonction de recherche vectorielle capable de filtrer
def _smart_vector_search(index: IndexStore, query: str, top_k: int, filter_dict: Dict = None) -> List[Tuple[str, float]]:
    emb_backend = index.embedding_backend or EmbeddingBackend()
    query_vec = emb_backend.encode([query])[0]

    # Appel à ChromaDB avec le paramètre 'where' pour le filtrage
    # C'est ici que le filtrage "form_code" se fait
    results = index.chroma.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=filter_dict
    )

    ids = results.get("ids", [[]])[0]
    dists = results.get("distances", [[]])[0]

    # Conversion distance -> score (1 - distance cosine)
    scores = [1 - d for d in dists]
    return list(zip(ids, scores))

# 2. Nouvelle méthode 'retrieve' pour la classe HybridRetriever
def smart_retrieve(self, query: str, manifest=None, top_k_sparse: int = None, top_k_dense: int = None) -> List[ContextualizedChunk]:
    k_sparse = top_k_sparse or config.BM25_TOP_K
    k_dense = top_k_dense or config.VECTOR_TOP_K

    # --- A. DÉTECTION DU CODE FORMULAIRE (ex: "IMM 5476") ---
    pattern = r"(IMM|CIT)\s?[-_]?\s?(\d{3,4})"
    match = re.search(pattern, query, re.IGNORECASE)

    specific_filter = None
    if match:
        # Normalisation : "imm5476" -> "IMM 5476"
        prefix = match.group(1).upper()
        number = match.group(2)
        target_code = f"{prefix} {number}"

        print(f"🎯 CIBLE DÉTECTÉE : {target_code} -> Filtrage strict activé.")
        specific_filter = {"form_code": target_code}

    # --- B. RECHERCHE VECTORIELLE AVEC FILTRE ---
    # On utilise notre nouvelle fonction _smart_vector_search définie plus haut
    vec_res = _smart_vector_search(self.index, query, k_dense, filter_dict=specific_filter)

    # --- C. RECHERCHE BM25 (LEXICALE) ---
    # BM25 ne filtre pas nativement, on doit filtrer les résultats après coup
    # On double le k pour avoir assez de candidats après filtrage
    bm25_res = _bm25_search(self.index, query, k_sparse * 2)

    if specific_filter:
        target = specific_filter["form_code"]
        # Filtrage manuel des résultats BM25 : on ne garde que ceux du bon formulaire
        filtered_bm25 = []
        for cid, score in bm25_res:
            # On récupère le chunk pour vérifier son code
            chunk = self.index.chunk_map.get(cid)
            if chunk and chunk.form_code == target:
                filtered_bm25.append((cid, score))
        bm25_res = filtered_bm25[:k_sparse]

    # --- D. FUSION ET CONTEXTE (Logique originale conservée) ---
    fused_ids = _rrf_fusion(bm25_res, vec_res)

    # Récupération des objets chunks complets
    chunk_candidates = []
    for cid, _ in fused_ids:
        if cid in self.index.chunk_map:
            chunk_candidates.append(self.index.chunk_map[cid])

    # Ajout du contexte (avant/après)
    contextualized = ContextualChunkEnhancer.enhance_with_context(chunk_candidates, manifest=manifest)
    return contextualized

# 3. Application du Patch (Monkey Patching)
HybridRetriever.retrieve = smart_retrieve
print("✅ Patch 'Smart Retrieval' appliqué avec succès !")

ImportError: cannot import name 'SearchResult' from 'rag_formulaire.retrieval' (/content/rag-formulaire/src/rag_formulaire/retrieval.py)

## 6) Poser des questions (sans CLI)

Le bloc suivant instancie les composants du pipeline et expose une fonction `ask_question` pour tester rapidement vos requêtes en français.


In [ ]:
from rag_formulaire import config
from rag_formulaire.evaluation import AdvancedSelfReflector, CRAGEvaluator, verify_response_against_evidence
from rag_formulaire.indexing import load_indexes
from rag_formulaire.llm import LocalLLM
from rag_formulaire.query_processing import AgenticQueryRouter, MultilingualQueryHandler, QueryDecomposer, QueryExpander
from rag_formulaire.reranker import CrossEncoderReranker
from rag_formulaire.retrieval import HybridRetriever

config.CHUNK_SIZE = 200
config.CHUNK_OVERLAP = 30  # Petit chevauchement pour garder le contexte



index_store = load_indexes()
query_handler = MultilingualQueryHandler()
router = AgenticQueryRouter()
expander = QueryExpander()
decomposer = QueryDecomposer()
retriever = HybridRetriever(index_store)
reranker = CrossEncoderReranker()
evaluator = CRAGEvaluator()
reflector = AdvancedSelfReflector()
llm = LocalLLM()


In [ ]:
def ask_question(question: str, evidence_k: int = config.FINAL_EVIDENCE_K):
    q_orig, q_fr = query_handler.normalize(question)
    route = router.route(q_fr)
    expansions = expander.expand(q_fr, n=3)
    subqueries = decomposer.decompose(q_fr) if route == "MULTI_STEP" else [q_fr]

    candidates = []
    for sub in subqueries:
        for variant in expansions:
            candidates.extend(retriever.retrieve(variant, manifest=None))

    reranked = reranker.rerank(q_fr, candidates, top_n=config.RERANK_TOP_N)
    if not reranked:
        return {"route": route, "answer": "Aucun extrait trouvé.", "evidence": []}

    scores = list(range(len(reranked), 0, -1))
    if not evaluator.is_evidence_strong(scores, reranked):
        return {"route": route, "answer": evaluator.fallback_message(), "evidence": []}

    evidence_texts = [
        f"[{c.base_chunk.form_code}] {c.base_chunk.section_title}: {c.base_chunk.content}"
        for c in reranked[:evidence_k]
    ]
    system_prompt = (
        "Vous êtes un assistant spécialisé dans les formulaires IRCC. Répondez uniquement en français en vous basant sur les "
        "extraits fournis. Citez le code du formulaire et la section."
    )
    user_prompt = q_fr + "Extraits:" + "".join(evidence_texts)
    answer = llm.chat(system_prompt, user_prompt, max_new_tokens=256)

    if verify_response_against_evidence(answer, reranked):
        answer = reflector.reflect(q_fr, answer, reranked)
    else:
        answer = evaluator.fallback_message()

    return {
        "route": route,
        "expansions": expansions,
        "answer": answer,
        "evidence": reranked[:evidence_k],
    }


In [ ]:
from IPython.display import display, Markdown

def display_result(result, manifest_list=None):
    """
    Affiche la réponse et les sources de manière formatée en Markdown.
    """
    # 1. En-tête avec la route utilisée
    md = f"### 🤖 Réponse (Stratégie : `{result['route']}`)\n\n"

    # 2. La réponse générée
    md += f"{result['answer']}\n\n"

    # 3. Les sources (Preuves)
    md += "---\n#### 🔍 Sources utilisées :\n"

    # Création d'un dictionnaire pour retrouver les URL à partir du code formulaire
    url_map = {m['form_code']: m['pdf_url'] for m in manifest_list} if manifest_list else {}

    for i, ev in enumerate(result['evidence'], 1):
        chunk = ev.base_chunk
        form_code = chunk.form_code
        section = chunk.section_title

        # Lien vers le PDF officiel si disponible
        if form_code in url_map:
            source_link = f"[{form_code}]({url_map[form_code]})"
        else:
            source_link = f"**{form_code}**"

        # Petit extrait du texte pour contexte (nettoyé des sauts de ligne)
        preview = chunk.content.replace("\n", " ")[:500] + "..."

        md += f"{i}. {source_link} — *{section}* (Page {chunk.page_number})\n"
        md += f"   > <small>{preview}</small>\n"

    display(Markdown(md))

In [ ]:
import torch
import gc

# 1. Force Python garbage collection
gc.collect()

# 2. Clear CUDA (GPU) cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU cache cleared.")

# 3. Check status
!nvidia-smi

In [ ]:
import torch
import gc

# Liste étendue de questions pour couvrir le manifest (22 questions)
test_questions = [
    # --- Identification des formulaires ---
    "À quoi sert le formulaire IMM 0008 Annexe 9 ?",
    "Quel est le code du formulaire pour la Déclaration d'union de fait ?",
    "Quel formulaire utiliser pour la déclaration d'un parent d'un mineur durant la COVID-19 ?",
    "Quel document est nécessaire pour le traitement de la syphilis ?",

    # --- Détails personnels et antécédents ---
    "Quelles maladies sont mentionnées dans le questionnaire sur l'état de santé (IMM 5955) ?",
    "Quel formulaire demande des détails sur les études, l'emploi et le déplacement (IMM 0104) ?",
    "Quel formulaire utiliser pour fournir des renseignements sur la famille (IMM 5707) ?",
    "Quel document remplir pour déclarer des détails de postes gouvernementaux ?",
    "Je suis un ancien policier, quel formulaire dois-je remplir pour donner les détails de mon service ?",

    # --- Représentation et Confidentialité ---
    "Qu'est-ce qu'un représentant rémunéré selon le formulaire IMM 5476 ?",
    "Quel formulaire utiliser pour autoriser la communication de renseignements à une personne désignée ?",
    "Quel document signer pour consentir à une demande d'accès à l'information (IMM 5744) ?",
    "Quel formulaire permet de faire une demande d'accès à des renseignements personnels (IMM 5563) ?",

    # --- Permis de travail et Aides familiaux ---
    "Quelle est la liste de contrôle des documents pour un permis de travail ?",
    "Quel formulaire remplir pour une demande de permis de travail présentée à l'extérieur du Canada ?",
    "Quel formulaire doit remplir un employeur particulier pour une aide à domicile (IMM 0267) ?",
    "Quel formulaire concerne les employeurs commerciaux pour les aides à domicile ?",
    "Quelle est la liste de contrôle pour les aides de soins à domicile (volet Travailleurs au Canada) ?",

    # --- Parrainage et Réfugiés ---
    "Quel formulaire de profil de réfugié est utilisé pour le parrainage d'aide conjointe ?",
    "Quel est le formulaire d'engagement pour le parrainage d'aide conjointe (IMM 1324) ?",
    "Qu'est-ce que l'évaluation du répondant (IMM 5492) ?",
    "En quoi consiste le plan d'aide à l'établissement (IMM 5494) ?",

    # --- Frais et Remboursements ---
    "Comment demander un renvoi des frais de traitement (IMM 5741) ?",
    "Quel formulaire concerne le renvoi de frais pour les immigrants investisseurs ?"
]

print(f"🚀 Lancement de la nouvelle série de {len(test_questions)} questions avec gestion mémoire...\n")

for i, question in enumerate(test_questions, 1):
    print(f"▶️ Question {i}/{len(test_questions)}: {question}")

    try:
        # Interrogation du pipeline
        result = ask_question(question)

        # Affichage propre
        # Assurez-vous que la variable 'manifest' (chargée à l'étape 5) est disponible
        display_result(result, manifest_list=manifest)

    except RuntimeError as e:
        if "out of memory" in str(e):
            print("⚠️ ERREUR OOM : Mémoire GPU saturée sur cette question.")
        else:
            print(f"⚠️ Erreur inattendue : {e}")

    print("\n" + "="*80 + "\n")

    # --- NETTOYAGE MÉMOIRE CRITIQUE ---
    # 1. Supprimer la référence aux résultats (qui peuvent contenir des tenseurs)
    if 'result' in locals():
        del result

    # 2. Forcer le Garbage Collector Python à libérer la RAM système
    gc.collect()

    # 3. Vider le cache de la mémoire vidéo (VRAM) du GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # ----------------------------------